# GPU Matrix Multiplication — NVIDIA RTX 4070 SUPER
Performs matrix multiplication on the GPU using **PyTorch** (CUDA), then verifies correctness against **NumPy** on the CPU.

PyTorch bundles its own CUDA runtime — no system-level CUDA install required.

In [ ]:
%pip install torch --index-url https://download.pytorch.org/whl/cu124

## 1. Check GPU

In [1]:
import torch
import numpy as np

assert torch.cuda.is_available(), "CUDA not available!"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")

GPU: NVIDIA GeForce RTX 4070 SUPER
VRAM: 11.6 GB
PyTorch version: 2.10.0+cu128
CUDA version: 12.8


## 2. Create Matrices

In [2]:
np.random.seed(42)

# Create two random matrices on the CPU (NumPy)
SIZE = 1024
A_cpu = np.random.rand(SIZE, SIZE).astype(np.float32)
B_cpu = np.random.rand(SIZE, SIZE).astype(np.float32)

print(f"Matrix A: {A_cpu.shape}, dtype={A_cpu.dtype}")
print(f"Matrix B: {B_cpu.shape}, dtype={B_cpu.dtype}")
print(f"Total elements per matrix: {SIZE*SIZE:,}")

Matrix A: (1024, 1024), dtype=float32
Matrix B: (1024, 1024), dtype=float32
Total elements per matrix: 1,048,576


## 3. GPU Matrix Multiplication (CuPy on RTX 4070)

In [3]:
import time

# Upload matrices to GPU (as PyTorch tensors)
A_gpu = torch.tensor(A_cpu, device="cuda")
B_gpu = torch.tensor(B_cpu, device="cuda")

# Warm-up pass (first CUDA call has JIT overhead)
_ = torch.matmul(A_gpu, B_gpu)
torch.cuda.synchronize()

# Timed GPU multiplication
start = time.perf_counter()
C_gpu = torch.matmul(A_gpu, B_gpu)
torch.cuda.synchronize()  # Wait for GPU to finish
gpu_time = time.perf_counter() - start

print(f"GPU result shape: {tuple(C_gpu.shape)}")
print(f"GPU time: {gpu_time*1000:.2f} ms")

GPU result shape: (1024, 1024)
GPU time: 0.24 ms


## 4. CPU Matrix Multiplication (NumPy) for Verification

In [4]:
# Timed CPU multiplication
start = time.perf_counter()
C_cpu = np.matmul(A_cpu, B_cpu)
cpu_time = time.perf_counter() - start

print(f"CPU result shape: {C_cpu.shape}")
print(f"CPU time: {cpu_time*1000:.2f} ms")
print(f"\nSpeedup: {cpu_time/gpu_time:.1f}x faster on GPU")

CPU result shape: (1024, 1024)
CPU time: 43.09 ms

Speedup: 180.5x faster on GPU


## 5. Verify Correctness

In [5]:
# Bring GPU result back to CPU for comparison
C_gpu_cpu = C_gpu.cpu().numpy()

max_diff = np.max(np.abs(C_gpu_cpu - C_cpu))
mean_diff = np.mean(np.abs(C_gpu_cpu - C_cpu))

print("=== Correctness Check ===")
print(f"Max absolute difference:  {max_diff:.2e}")
print(f"Mean absolute difference: {mean_diff:.2e}")

# float32 has ~7 decimal digits of precision; differences < 1e-3 are expected and fine
tolerance = 1e-3
match = np.allclose(C_gpu_cpu, C_cpu, atol=tolerance)
print(f"\nnp.allclose(atol={tolerance}): {'PASS — GPU and CPU results match!' if match else 'FAIL'}")

print("\n=== Spot Check (top-left 3x3 corner) ===")
print("GPU:\n", C_gpu_cpu[:3, :3])
print("CPU:\n", C_cpu[:3, :3])

=== Correctness Check ===
Max absolute difference:  2.14e-04
Mean absolute difference: 3.55e-05

np.allclose(atol=0.001): PASS — GPU and CPU results match!

=== Spot Check (top-left 3x3 corner) ===
GPU:
 [[248.73703 248.625   246.43938]
 [262.98972 253.95505 252.4527 ]
 [256.46448 259.9904  255.3051 ]]
CPU:
 [[248.73706 248.62498 246.43936]
 [262.9897  253.95497 252.45267]
 [256.46445 259.99036 255.30502]]


## 6. Small Hand-Verifiable Example
Cross-check with a tiny matrix whose result you can compute by hand.

In [6]:
# Simple 2x2 example: easy to verify by hand
# A = [[1, 2],   B = [[5, 6],   A@B = [[1*5+2*7, 1*6+2*8],  = [[19, 22],
#      [3, 4]]        [7, 8]]          [3*5+4*7, 3*6+4*8]]       [43, 50]]

A_small = np.array([[1, 2], [3, 4]], dtype=np.float32)
B_small = np.array([[5, 6], [7, 8]], dtype=np.float32)
expected = np.array([[19, 22], [43, 50]], dtype=np.float32)

gpu_result = torch.matmul(
    torch.tensor(A_small, device="cuda"),
    torch.tensor(B_small, device="cuda")
).cpu().numpy()
cpu_result = np.matmul(A_small, B_small)

print("A =\n", A_small)
print("B =\n", B_small)
print("\nExpected  A @ B =\n", expected)
print("GPU       A @ B =\n", gpu_result)
print("CPU       A @ B =\n", cpu_result)
print(f"\nGPU matches expected: {'PASS' if np.array_equal(gpu_result, expected) else 'FAIL'}")
print(f"CPU matches expected: {'PASS' if np.array_equal(cpu_result, expected) else 'FAIL'}")

A =
 [[1. 2.]
 [3. 4.]]
B =
 [[5. 6.]
 [7. 8.]]

Expected  A @ B =
 [[19. 22.]
 [43. 50.]]
GPU       A @ B =
 [[19. 22.]
 [43. 50.]]
CPU       A @ B =
 [[19. 22.]
 [43. 50.]]

GPU matches expected: PASS
CPU matches expected: PASS
